# Data Preprocessing

This notebook prepares the raw dataset for machine learning model training.

The preprocessing steps are based on the findings and decisions made during the exploratory data analysis.

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [28]:
# df = pd.read_csv('/content/signal_metrics.csv')
df = pd.read_csv('../data/raw/signal_metrics.csv')

In [29]:
df.head()

,Timestamp,Locality,Latitude,Longitude,Signal Strength (dBm),Signal Quality (%),Data Throughput (Mbps),Latency (ms),Network Type,BB60C Measurement (dBm),srsRAN Measurement (dBm),BladeRFxA9 Measurement (dBm)
0,2023-05-05 12:50:40.000000,Anisabad,25.599109,85.137355,-84.274113,0.0,1.863890,129.122914,3G,0.000000,0.000000,0.000000
1,2023-05-05 12:53:47.210173,Fraser Road,25.433286,85.070053,-97.653121,0.0,5.132296,54.883606,4G,-95.810791,-105.452359,-99.920892
2,2023-05-05 12:56:54.420346,Boring Canal Road,25.498809,85.211371,-87.046134,0.0,1.176985,119.598286,LTE,-91.593861,-95.419482,-87.714070
3,2023-05-05 13:00:01.630519,Danapur,25.735138,85.208400,-94.143159,0.0,68.596932,46.598387,5G,-90.642773,-101.895905,-96.570698
4,2023-05-05 13:03:08.840692,Phulwari Sharif,25.538556,85.159860,-94.564765,0.0,38.292038,30.342828,5G,-90.489100,-103.318304,-95.102467


## 2. Remove Unnecessary Features

Based on the EDA findings, several features were removed before model training.

- `Timestamp` was removed because no clear temporal pattern was observed.
- `Signal Quality (%)` was removed because it is a constant feature with zero variance.
- `BB60C Measurement (dBm)`, `srsRAN Measurement (dBm)`, and `BladeRFxA9 Measurement (dBm)` were removed because they provide highly redundant information.

The target variable, `Data Throughput (Mbps)`, is retained for the prediction task.

In [30]:
df_processed = df.copy()

In [31]:
columns_to_drop = [
    'Timestamp',
    'Signal Quality (%)',
    'BB60C Measurement (dBm)',
    'srsRAN Measurement (dBm)',
    'BladeRFxA9 Measurement (dBm)'
]

df_processed = df_processed.drop(columns = columns_to_drop)

df_processed.head()

,Locality,Latitude,Longitude,Signal Strength (dBm),Data Throughput (Mbps),Latency (ms),Network Type
0,Anisabad,25.599109,85.137355,-84.274113,1.863890,129.122914,3G
1,Fraser Road,25.433286,85.070053,-97.653121,5.132296,54.883606,4G
2,Boring Canal Road,25.498809,85.211371,-87.046134,1.176985,119.598286,LTE
3,Danapur,25.735138,85.208400,-94.143159,68.596932,46.598387,5G
4,Phulwari Sharif,25.538556,85.159860,-94.564765,38.292038,30.342828,5G


In [32]:
df_processed['Network Type'] = df_processed['Network Type'].replace({
    'LTE': '4G'
})

In [33]:
df_processed['Network Type'].value_counts()

Network Type
4G    8443
3G    4208
5G    4178
Name: count, dtype: int64

In [34]:
df_processed.columns

Index(['Locality', 'Latitude', 'Longitude', 'Signal Strength (dBm)',
       'Data Throughput (Mbps)', 'Latency (ms)', 'Network Type'],
      dtype='str')

## 3. Separate Features and Target

The target variable is **Data Throughput (Mbps)**, which is the continuous value we want to predict.

The remaining columns are used as input features for the machine learning models.

In [35]:
X = df_processed.drop(columns='Data Throughput (Mbps)')
y = df_processed['Data Throughput (Mbps)']

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')

X shape : (16829, 6)
y shape : (16829,)


## 4. Train/Test Split

The dataset was divided into training and testing sets using an 80/20 split.

The training set will be used to train the machine learning models, while the testing set will be reserved for evaluating their performance on unseen data.

A fixed random state is used to ensure reproducibility.

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state = 42
)

print(f'X_train shape {X_train.shape}')
print(f'X_test shape {X_test.shape}')
print(f'y_train shape {y_train.shape}')
print(f'y_test shape {y_test.shape}')

X_train shape (13463, 6)
X_test shape (3366, 6)
y_train shape (13463,)
y_test shape (3366,)


## 5. Save the Train/Test Sets

The training and testing sets are saved separately in the Colab working directory.

Saving the datasets separately allows the same train/test split to be reused across different modeling notebooks while maintaining consistent and reproducible model evaluation.

The datasets are saved before scaling and categorical encoding. These preprocessing steps will be applied later using the training data only to prevent data leakage.


In [43]:
# os.makedirs('../data/processed', exist_ok=True)
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)

y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)
df_processed.to_csv('../data/processed/processed_data.csv', index = False)

print("Train/test sets saved successfully.")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Train/test sets saved successfully.
X_train: (13463, 6)
X_test: (3366, 6)
y_train: (13463,)
y_test: (3366,)


In [44]:
X_train.head()

,Locality,Latitude,Longitude,Signal Strength (dBm),Latency (ms),Network Type
12573,Pataliputra,25.605539,85.285651,-87.611830,164.101054,4G
5139,Kidwaipuri,25.491973,85.086669,-90.089704,22.851745,5G
5900,Danapur,25.684972,84.989785,-90.175391,139.812003,4G
3328,Kankarbagh,25.569868,85.138714,-93.384826,113.904981,4G
8457,Phulwari Sharif,25.626811,85.136823,-95.888704,71.291449,4G
